In [0]:
# ═══════════════════════════════════════════════════════════════
# CLARITY AML — Shared Utilities
# ═══════════════════════════════════════════════════════════════

STORAGE_ACCOUNT = "clarityadls"

# ── Path helpers ───────────────────────────────────────────────
def bronze(path=""):
    return f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/{path}"

def silver(path=""):
    return f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/{path}"

def gold(path=""):
    return f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/{path}"

# ── ADLS connection setup ──────────────────────────────────────
def setup_adls():
    CLIENT_ID     = dbutils.secrets.get("clarity-aml-secrets", "azure-client-id")
    CLIENT_SECRET = dbutils.secrets.get("clarity-aml-secrets", "azure-client-secret")
    TENANT_ID     = dbutils.secrets.get("clarity-aml-secrets", "azure-tenant-id")

    spark.conf.set(
        f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        "OAuth"
    )
    spark.conf.set(
        f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )
    spark.conf.set(
        f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        CLIENT_ID
    )
    spark.conf.set(
        f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        CLIENT_SECRET
    )
    spark.conf.set(
        f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token"
    )
    print(f"✅ ADLS Gen2 connected: {STORAGE_ACCOUNT}")

# ── Read helpers ───────────────────────────────────────────────
def read_silver_transactions():
    return spark.read.parquet(silver("transactions"))

def read_bronze_sanctions_ofac():
    return spark.read.csv(
        bronze("reference/sanctions/ofac_sdn.csv"),
        header=False,
        inferSchema=False
    )

def read_bronze_kvk():
    return spark.read.csv(
        bronze("reference/kvk/kvk_companies.csv"),
        header=True,
        inferSchema=True
    )

# ── Write helpers ──────────────────────────────────────────────
def write_silver(df, path, partition_cols=["value_date_year", "value_date_month", "value_date_day"]):
    df.write \
      .mode("overwrite") \
      .partitionBy(*partition_cols) \
      .parquet(silver(path))
    print(f"✅ Written to Silver: {path}")

def write_gold(df, path, partition_cols=None):
    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(gold(path))
    print(f"✅ Written to Gold: {path}")

print("✅ Clarity AML utilities loaded")


✅ Clarity AML utilities loaded


In [0]:
SNOWFLAKE_OPTIONS = {
    "sfURL":       "ottwzhp-oa24595.snowflakecomputing.com",
    "sfUser":      "vedoxo123",
    "sfPassword":  "Vedshraddha@1234",
    "sfDatabase":  "CLARITY_AML",
    "sfSchema":    "GOLD",
    "sfWarehouse": "CLARITY_WH"
}

# ── Test connection by reading Snowflake system metadata ───────
try:
    test_df = spark.read \
        .format("snowflake") \
        .options(**SNOWFLAKE_OPTIONS) \
        .option("query", "SELECT CURRENT_USER(), CURRENT_DATABASE(), CURRENT_WAREHOUSE()") \
        .load()

    row = test_df.collect()[0]
    print("✅ Snowflake connection successful!")
    print(f"   User:      {row[0]}")
    print(f"   Database:  {row[1]}")
    print(f"   Warehouse: {row[2]}")

except Exception as e:
    print(f"❌ Snowflake connection failed: {e}")